In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
from rag_helper import RAGBase

instructions = """

You are a course teaching assistant.
Answer the question based on CONTEXT from the FAQ Database.
Use only the facts from the CONTEXT when answering the question.
""".strip()

assistant = RAGBase(
    index= index,
    llm_client=openai_client,
    instructions=instructions,
)

In [ ]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

In [ ]:
messages = [
    {'role' : 'user', 'content': 'I just discovered the course, can I join it?'
    }
]

response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
)

response.output_text

In [ ]:
def search(query):
    boost_dict = {'question' : 3.0, 'section' : 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict= boost_dict,
        filter_dict= filter_dict
    )

In [ ]:
index.search("How to run ollama?")

In [ ]:
search_tool = {
    'type' : 'function',
    'name' : 'search',
    'description' : 'Search the FAQ database for entries matching the given query.',
    'parameters' : {
        'type' :'object',
        'properties' :{
            'query' : {
                'type' : 'string',
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        'required':['query'],
        'additionalProperties' : False
    }
}

In [ ]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input = messages,
    tools = [search_tool]

)

In [ ]:
len(response.output)

In [ ]:
call = response.output[0]

In [ ]:
call

In [ ]:
import json

args = json.loads(call.arguments)

In [ ]:
results = search(**args)

In [ ]:
result_json = json.dumps(results, indent=2)

In [ ]:
print(result_json)

In [ ]:
function_call_output = {
    'type':'function_call_output',
    'call_id' : call.call_id,
    'output' : result_json
}

In [ ]:
messages.append(call)

In [ ]:
messages.append(function_call_output)

In [ ]:
messages

In [ ]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [ ]:
print(response.output_text)

In [ ]:
usage = response.usage

usage.input_tokens, usage.output_tokens